# Distributed Weather Data Analysis using MapReduce

This notebook demonstrates how to process large-scale weather data to find the hottest and coolest years using the MapReduce paradigm.

In [7]:
!pip install mrjob

### 1. Data Acquisition
We will download a subset of historical weather data. For this example, we will fetch data containing year and temperature observations.

In [12]:
import requests
import pandas as pd
import random

API_KEY = "645caeaa7f76662b5f2ce1261a0ed334"
cities = ["London", "New York", "Tokyo", "Delhi", "Paris", "Sydney", "Berlin", "Cairo", "Moscow", "Mumbai"]
weather_records = []

print("Fetching live weather data...")
for city in cities:
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid=645caeaa7f76662b5f2ce1261a0ed334&units=metric"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            # Simulating different years for the MapReduce demonstration
            simulated_year = random.randint(2010, 2024)
            temp = data['main']['temp']
            weather_records.append([simulated_year, temp])
        else:
            print(f"Failed to fetch data for {city}: Status {response.status_code}")
    except Exception as e:
        print(f"Error fetching data for {city}: {e}")

# Create the CSV from real API data
if weather_records:
    df_real = pd.DataFrame(weather_records, columns=['year', 'temp'])
    df_real.to_csv('weather_data.csv', index=False, header=False)
    print(f"Successfully fetched {len(weather_records)} records and updated weather_data.csv")
    display(df_real.head())
else:
    print("No data fetched. Check API key or connection.")

Fetching live weather data...
Successfully fetched 10 records and updated weather_data.csv


,year,temp
0,2013,9.38
1,2012,0.99
2,2010,15.93
3,2017,24.57
4,2011,11.74


### 2. MapReduce Implementation
The Mapper will emit `(year, temperature)` pairs. The Reducer will calculate the average temperature for each year.

In [13]:
%%writefile weather_analysis.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MRWeatherAnalysis(MRJob):

    def mapper(self, _, line):
        # Data format: year,temp
        try:
            fields = line.split(',')
            if len(fields) == 2:
                year = fields[0]
                temp = float(fields[1])
                yield year, temp
        except ValueError:
            pass

    def reducer(self, year, temps):
        temp_list = list(temps)
        avg_temp = sum(temp_list) / len(temp_list)
        yield None, (avg_temp, year)

    def reducer_find_extremes(self, _, year_stats):
        stats = list(year_stats)
        yield "Hottest Year", max(stats)
        yield "Coolest Year", min(stats)

    def steps(self):
        return [
            MRStep(mapper=self.mapper, reducer=self.reducer),
            MRStep(reducer=self.reducer_find_extremes)
        ]

if __name__ == '__main__':
    MRWeatherAnalysis.run()

Overwriting weather_analysis.py


### 3. Execution
Now we run the MapReduce job locally.

In [14]:
!python weather_analysis.py weather_data.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/weather_analysis.root.20260408.072254.810683
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/weather_analysis.root.20260408.072254.810683/output
Streaming final output from /tmp/weather_analysis.root.20260408.072254.810683/output...
"Hottest Year"	[27.91, "2024"]
"Coolest Year"	[2.34, "2019"]
Removing temp directory /tmp/weather_analysis.root.20260408.072254.810683...
